# Stage 2 finalists, part B

Finalists from the Stage 2 screen, trained with the Stage 1A protocol (FNO, lr $10^{-3}$, MSE,
20 epochs, batch 32, basis refresh every 200 steps). Test predictions are saved for the Stage 3 ensembles.
Adam with the same protocol is Stage 1A and is not rerun.

| optimizer | seed |
|---|---|
| adaptive tau=0.99 | 0 |
| adaptive tau=0.99 | 1 |
| adaptive tau=0.99 | 2 |
| GaLore r=4 | 0 |
| GaLore r=4 | 1 |
| GaLore r=4 | 2 |
| GaLore r=16 | 1 |
| GaLore r=16 | 2 |

## Setup

In [ ]:
import sys, glob, os, importlib.util
if importlib.util.find_spec("neuralop") is None:
    !pip install -q neuraloperator
# project code (nsbench, optim, solver) comes from the private ns-surrogate-code dataset
hits = glob.glob("/kaggle/input/**/nsbench/__init__.py", recursive=True)
code_dir = (os.path.dirname(os.path.dirname(hits[0])) if hits else
            next(os.path.abspath(d) for d in (".", "..") if os.path.isdir(f"{d}/nsbench")))
sys.path.insert(0, code_dir)
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nsbench
from nsbench import Trainer, save_csv, save_fig, save_predictions, run_record
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.DataFrame([{"python": sys.version.split()[0], "torch": torch.__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "code": code_dir}])
from optim import LowRankAdam

## Data

In [ ]:
D = nsbench.load()
pd.DataFrame([{"data": D.data_dir, "train batches": len(D.train_loader), "val batches": len(D.val_loader),
               "test samples": len(D.x_test), "y_mean": D.y_mean.item(), "y_std": D.y_std.item()}])

## Runs

In [ ]:
EPOCHS, LR, REFRESH = 20, 1e-3, 200
CFGS = {'GaLore r=16': {'rank_mode': 'fixed', 'rank': 16}, 'GaLore r=8': {'rank_mode': 'fixed', 'rank': 8}, 'GaLore r=4': {'rank_mode': 'fixed', 'rank': 4}, 'adaptive tau=0.999': {'rank_mode': 'adaptive', 'tau': 0.999, 'r_min': 2, 'r_max': 64}, 'adaptive tau=0.99': {'rank_mode': 'adaptive', 'tau': 0.99, 'r_min': 2, 'r_max': 64}}
RUNS = [('adaptive tau=0.99', 0), ('adaptive tau=0.99', 1), ('adaptive tau=0.99', 2), ('GaLore r=4', 0), ('GaLore r=4', 1), ('GaLore r=4', 2), ('GaLore r=16', 1), ('GaLore r=16', 2)]
pd.DataFrame([{"optimizer": n, "seed": s, **CFGS[n]} for n, s in RUNS])

## Training

In [ ]:
rows, hists, ranklogs = [], [], []
for name, seed in RUNS:
    cfg = CFGS[name]
    tr = Trainer(nsbench.FourierNeuralOperator, lambda p, cfg=cfg: LowRankAdam(p, lr=LR, refresh=REFRESH, **cfg),
                 D.train_loader, D.val_loader, D.test_loader, D.y_mean, D.y_std, device,
                 seed=seed, label=f"FNO {name} seed {seed}").fit(EPOCHS)
    rows.append(run_record(tr.summary(), run="FNO", optimizer=name, lr=LR, seed=seed, epochs=EPOCHS, batch=32,
                           loss_fn="MSE", monitor="loss", refresh=REFRESH, **cfg))
    hists.append(tr.history().assign(optimizer=name, seed=seed))
    ranklogs.append(pd.DataFrame(tr.opt.rank_log, columns=["step", "param", "rank", "energy"]).assign(optimizer=name, seed=seed))
    save_predictions(tr, D.x_test, f"fno_{name}_seed{seed}")
    # save after every run so a timeout keeps finished runs
    save_csv(pd.concat(rows, ignore_index=True), "stage2_final_b")
    save_csv(pd.concat(hists, ignore_index=True), "stage2_final_b_history")
    save_csv(pd.concat(ranklogs, ignore_index=True), "stage2_final_b_ranks")
    del tr; torch.cuda.empty_cache()
final = pd.concat(rows, ignore_index=True)
final[["optimizer", "seed", "best_epoch", "test_rel_L2", "opt_state_MB", "peak_gpu_MB", "sec_per_epoch"]]

## Mean ± std per optimizer

In [ ]:
final.groupby("optimizer")[["test_rel_L2", "test_mse_norm", "opt_state_MB", "peak_gpu_MB", "sec_per_epoch"]].agg(["mean", "std", "count"])